# AFS Parse

Runs the full extraction pipeline on one or more filings in `COMMON.PDF_STAGING`.

**Steps:**
1. Select a filing from `PDF_STAGING` (status = `pending`)
2. Provide org details if this is a new organization
3. Run `pipeline.process_filing()` — identify → classify → extract → load
4. On success, status is set to `done`; on failure, `failed`

In [ ]:
import sys, json
sys.path.insert(0, '../src')

from afs import pipeline

In [ ]:
# Show pending filings
import pandas as pd

df = session.sql("""
    SELECT STAGING_ID, FILENAME, TOTAL_PAGES, STATUS, EXTRACTED_AT
      FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING
     WHERE STATUS IN ('pending', 'failed')
     ORDER BY EXTRACTED_AT
""").to_pandas()

print(f'{len(df)} filing(s) ready to parse:')
df

In [ ]:
# ── CONFIGURE ─────────────────────────────────────────────────────────────────
# Set STAGING_ID to the value from the table above, or None to process all pending.
STAGING_ID  = None          # e.g. '3f8a...' or None for all pending

# For a NEW organization, provide org details. For known orgs (matched by EIN or name),
# leave as None — the pipeline will auto-match from ORG_REGISTRY.
ORG_HINT    = None          # e.g. {'org_code': 'ACME_HEALTH', 'legal_name': 'Acme Health System'}

REPARSE     = False         # Set True to re-extract even if filing_id already loaded
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# Load staging rows to process
if STAGING_ID:
    where = f"WHERE STAGING_ID = '{STAGING_ID}'"
else:
    where = "WHERE STATUS IN ('pending', 'failed')"

rows = session.sql(f"""
    SELECT STAGING_ID, FILENAME, FILING_ID, TOTAL_PAGES,
           PAGE_TEXTS::STRING AS PAGE_TEXTS_STR
      FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING
      {where}
     ORDER BY EXTRACTED_AT
""").collect()

print(f'Processing {len(rows)} filing(s)...')

In [ ]:
# Run the pipeline
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s %(message)s')

for row in rows:
    staging_id  = row[0]
    filename    = row[1]
    filing_id   = row[2]
    total_pages = row[3]
    page_texts  = json.loads(row[4])

    print(f'\n=== {filename} ({total_pages} pages) ===')

    # Mark in-progress
    session.sql(f"""
        UPDATE AUDITED_FINANCIALS.COMMON.PDF_STAGING
           SET STATUS = 'processing'
         WHERE STAGING_ID = '{staging_id}'
    """).collect()

    try:
        report = pipeline.process_filing(
            session,
            filename=filename,
            filing_id=filing_id,
            page_texts=page_texts,
            total_pages=total_pages,
            org_hint=ORG_HINT,
            reparse=REPARSE,
        )
        session.sql(f"""
            UPDATE AUDITED_FINANCIALS.COMMON.PDF_STAGING
               SET STATUS = 'done'
             WHERE STAGING_ID = '{staging_id}'
        """).collect()
        print(json.dumps(report.get('stages', {}), indent=2, default=str))
        if report.get('skipped'):
            print(f'[skip] {report["skipped"]}')
        else:
            print('[ok]')
    except Exception as exc:
        session.sql(f"""
            UPDATE AUDITED_FINANCIALS.COMMON.PDF_STAGING
               SET STATUS = 'failed'
             WHERE STAGING_ID = '{staging_id}'
        """).collect()
        print(f'[fail] {exc}')
        raise